# MegaRAG: Multimodal Knowledge Graph-based RAG on Kaggle

This notebook reproduces **MegaRAG** on Kaggle using the bundled repository [`reproduce_MegaRAG`](https://github.com/linhnguyen15492/reproduce_MegaRAG.git), which contains the complete source code for **MegaRAG**, **LightRAG**, and **MinerU**.

MegaRAG enables **global visual question answering** on documents by constructing a **Multimodal Knowledge Graph (MMKG)** combining graph-based reasoning with document page retrieval.

### Prerequisites
- **GPU Accelerator**: Tesla T4 / P100 or better (enable GPU in Kaggle settings)
- **Internet**: Turned ON in Kaggle settings
- **OpenAI API Key**: Added to Kaggle Secrets as `OPENAI_API_KEY` (or entered interactively)

**Paper**: [MegaRAG: Multimodal Graph-based Retrieval Augmented Generation (ACL 2026)](https://arxiv.org/abs/2512.20626)


## 1. System Setup & Clone Repository


In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path
import torch

# Force PyTorch only & avoid CUDA memory fragmentation
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set base working directory (default Kaggle working directory is /kaggle/working)
if Path("/kaggle/working").exists():
    BASE_DIR = Path("/kaggle/working")
else:
    BASE_DIR = Path.cwd()

# Clone or locate the reproduce_MegaRAG repository
REPO_URL = "https://github.com/linhnguyen15492/reproduce_MegaRAG.git"
REPO_NAME = "reproduce_MegaRAG"

if (BASE_DIR / "MegaRAG").exists() and (BASE_DIR / "MinerU").exists():
    REPO_DIR = BASE_DIR
elif (BASE_DIR / REPO_NAME / "MegaRAG").exists():
    REPO_DIR = BASE_DIR / REPO_NAME
else:
    print(f"Cloning repository {REPO_URL} into {BASE_DIR}...")
    subprocess.run(f"git clone {REPO_URL}", shell=True, check=True, cwd=BASE_DIR)
    REPO_DIR = BASE_DIR / REPO_NAME

os.chdir(REPO_DIR)
print(f"✓ Working directory: {REPO_DIR}")

# Set component paths from the repository
mineru_dir = REPO_DIR / "MinerU"
megarag_dir = REPO_DIR / "MegaRAG"
lightrag_dir = REPO_DIR / "LightRAG"


## 2. Install Required Dependencies & Local Packages


In [ ]:
import subprocess
import sys

# 1. Install base dependencies for MinerU, LightRAG, and MegaRAG
required_packages = [
    "pyopenssl>=24.0.0",              # Fix Kaggle OpenSSL/cryptography mismatch
    "cryptography>=42.0.0",          # Fix GEN_EMAIL attribute error
    "transformers",
    "pillow>=10.2.0,<11.0.0",        # Avoid Pillow 11 typing issues with RapidTable
    "PyMuPDF==1.24.14",              # Required by MinerU/magic-pdf (<1.25.0)
    "pdfminer.six==20231228",        # Required by MinerU/magic-pdf
    "pypdfium2",                     # PDF rendering
    "rapid_table==1.0.3",            # Required for table extraction
    "loguru",                        # Logging
    "boto3",                         # MinerU dependency
    "timm",                          # Vision backbones
    "einops",                        # Tensor operations
    "openai>=1.50.0,<2.0.0",         # Modern OpenAI SDK (v1.x)
    "accelerate>=0.30.0,<2.0.0",     # PyTorch acceleration
    "beautifulsoup4>=4.12.0",        # HTML/XML parsing
    "opencv-python-headless",        # Headless OpenCV for server/Kaggle environments
    "ultralytics",                   # YOLO layout detection
    "doclayout-yolo",                # Document layout analysis
    "ftfy",                          # Text normalization
    "dill",                          # Serialization
    "shapely",                       # Bounding box geometry
    "pyclipper",                     # Polygon clipping
    "tiktoken",                      # Token counting for LLM
    "huggingface_hub",               # HuggingFace model download
    "matplotlib",                    # Visualization
    "rich",                          # Formatted console output
    "pyyaml",                        # YAML configuration parser
    "networkx",                      # Knowledge graph structures
]

print("Installing dependencies...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts"] + required_packages)
    print("✓ Base dependencies installed successfully!")
except Exception as e:
    print(f"⚠️ Batch install note ({e}), installing individual packages...")
    for package in required_packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        except Exception:
            pass

# 2. Install bundled packages from repository in editable mode
print("Installing MinerU from repository...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(mineru_dir)])

print("Installing LightRAG from repository...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(lightrag_dir)])

print("Installing MegaRAG from repository...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(megarag_dir), "--no-deps"])

print("✓ All dependencies and packages installed successfully!")


## 3. Setup MinerU Models & Configuration


In [ ]:
import os
import shutil
import json as json_lib
import yaml
from pathlib import Path
import torch
from huggingface_hub import snapshot_download

print("Downloading MinerU model weights from HuggingFace...")

# 1. Download PDF-Extract-Kit models
pdf_extract_kit_path = snapshot_download(
    repo_id="opendatalab/PDF-Extract-Kit-1.0",
    allow_patterns=["models/*"],
)
models_dir = Path(pdf_extract_kit_path) / "models"
print(f"✓ PDF-Extract-Kit models: {models_dir}")

# 2. Download LayoutReader model
layoutreader_path = snapshot_download(
    repo_id="hantian/layoutreader",
)
layoutreader_model_dir = Path(layoutreader_path)
print(f"✓ LayoutReader model: {layoutreader_model_dir}")

# 3. Setup OCR model weights and configurations
ocr_models_dir = models_dir / "OCR" / "paddleocr_torch"
if ocr_models_dir.exists():
    v3_multi_det = ocr_models_dir / "Multilingual_PP-OCRv3_det_infer.pth"
    v4_ch_rec = ocr_models_dir / "ch_PP-OCRv4_rec_infer.pth"
    v4_server_rec = ocr_models_dir / "ch_PP-OCRv4_rec_server_infer.pth"
    
    # Setup Detection weights
    if v3_multi_det.exists():
        for det_target in ["ch_PP-OCRv3_det_infer.pth", "en_PP-OCRv3_det_infer.pth", "ch_PP-OCRv4_det_infer.pth"]:
            dest = ocr_models_dir / det_target
            shutil.copy2(v3_multi_det, dest)
            print(f"✓ Configured OCR detection model: {det_target}")
    
    # Setup Server Rec alias if needed
    if v4_ch_rec.exists() and not v4_server_rec.exists():
        shutil.copy2(v4_ch_rec, v4_server_rec)

    # Configure models_config.yml in MinerU so 'en' maps to matching 6625-vocab recognition model
    for cfg_yml in mineru_dir.glob("**/models_config.yml"):
        try:
            with open(cfg_yml, "r", encoding="utf-8") as f:
                y_data = yaml.safe_load(f)
            if "lang" in y_data and "en" in y_data["lang"]:
                y_data["lang"]["en"]["rec"] = "ch_PP-OCRv4_rec_infer.pth"
                y_data["lang"]["en"]["dict"] = "ppocr_keys_v1.txt"
            if "lang" in y_data and "latin" in y_data["lang"]:
                y_data["lang"]["latin"]["rec"] = "ch_PP-OCRv4_rec_infer.pth"
                y_data["lang"]["latin"]["dict"] = "ppocr_keys_v1.txt"
            with open(cfg_yml, "w", encoding="utf-8") as f:
                yaml.dump(y_data, f)
            print(f"✓ Configured {cfg_yml.name} with consistent OCR model shapes")
        except Exception as e:
            pass

# 4. Generate magic-pdf.json / mineru.json configuration
device_mode = "cuda" if torch.cuda.is_available() else "cpu"
config_data = {
    "models-dir": str(models_dir),
    "device-mode": device_mode,
    "layoutreader-model-dir": str(layoutreader_model_dir),
    "layout-config": {
        "model": "doclayout_yolo"
    },
    "formula-config": {
        "enable": False
    },
    "table-config": {
        "model": "rapid_table",
        "sub_model": "slanet_plus",
        "enable": True,
        "max_time": 400
    },
    "latex-delimiter-config": {
        "display": {"left": "$$", "right": "$$"},
        "inline": {"left": "$", "right": "$"},
    },
}

config_paths = [
    Path.home() / "magic-pdf.json",
    Path("/root/magic-pdf.json"),
    Path.home() / "mineru.json",
    Path("/root/mineru.json"),
    mineru_dir / "magic-pdf.json",
    mineru_dir / "mineru.json",
    REPO_DIR / "magic-pdf.json",
]

for cfg_path in config_paths:
    try:
        cfg_path.parent.mkdir(parents=True, exist_ok=True)
        with open(cfg_path, "w", encoding="utf-8") as f:
            json_lib.dump(config_data, f, indent=4)
    except Exception:
        pass

print(f"✓ MinerU configuration generated successfully (device-mode: '{device_mode}')")


## 4. Setup API Keys & Environment Variables


In [ ]:
import os
import sys
import shutil
import re
import getpass
from pathlib import Path

# 1. Retrieve OpenAI API Key from Kaggle Secrets, Environment, or User Input
openai_api_key = os.environ.get("OPENAI_API_KEY")

if not openai_api_key:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        openai_api_key = user_secrets.get_secret("OPENAI_API_KEY")
        print("✓ OpenAI API Key loaded from Kaggle Secrets.")
    except Exception:
        openai_api_key = None

if not openai_api_key:
    openai_api_key = getpass.getpass("Enter your OpenAI API Key: ")

os.environ["OPENAI_API_KEY"] = openai_api_key

# 2. Update PATH with Python and MinerU bin locations
py_bin_dir = str(Path(sys.executable).parent)
extra_paths = [
    py_bin_dir,
    "/root/.local/bin",
    str(Path.home() / ".local" / "bin"),
    "/usr/local/bin",
    "/opt/conda/bin",
]
for p in extra_paths:
    if p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"

# 3. Write env.sh in MegaRAG directory
magic_pdf_bin = shutil.which("magic-pdf") or shutil.which("mineru") or py_bin_dir
magic_pdf_bin_dir = str(Path(magic_pdf_bin).parent) if magic_pdf_bin else str(Path(sys.executable).parent)

env_sh_content = f"""#!/usr/bin/env bash
export OPENAI_API_KEY="{openai_api_key}"
export MINERU_PATH="{magic_pdf_bin_dir}"
"""

env_file = megarag_dir / "env.sh"
with open(env_file, "w", encoding="utf-8") as f:
    f.write(env_sh_content)
print(f"✓ Configured {env_file}")

# 4. Runtime patches & Kaggle GPU memory optimizations
# A. Patch transformers configuration_auto to register hgnet_v2 globally
try:
    import transformers.models.auto.configuration_auto as tf_ca
    ca_file = Path(tf_ca.__file__)
    if ca_file.exists():
        with open(ca_file, "r", encoding="utf-8") as f:
            ca_code = f.read()
        if "hgnet_v2" not in ca_code:
            patch_reg = '''
# Auto register hgnet_v2 for MinerU
try:
    from mineru.model.layout.pp_doclayoutv2 import HGNetV2Config
    CONFIG_MAPPING.register("hgnet_v2", HGNetV2Config)
except Exception:
    pass
'''
            ca_code += "\n" + patch_reg
            with open(ca_file, "w", encoding="utf-8") as f:
                f.write(ca_code)
            print("✓ Patched transformers configuration_auto for hgnet_v2")
except Exception as e:
    pass

# B. Patch MegaRAG construct_mmkg.py & query_mmkg.py for GME require_version bypass
for py_name in ["construct_mmkg.py", "query_mmkg.py"]:
    script_path = megarag_dir / "egs" / "utils" / py_name
    if script_path.exists():
        with open(script_path, "r", encoding="utf-8") as f:
            code = f.read()
        if "versions.require_version" not in code:
            code = code.replace(
                "def initialize_model():",
                "def initialize_model():\n    try:\n        import transformers.utils.versions as versions\n        versions.require_version = lambda *args, **kwargs: None\n    except Exception:\n        pass"
            )
        if "llm_max_async" not in code and "llm_model_max_async" not in code:
            code = code.replace(
                "rag = MegaRAG(\n        working_dir=str(working_dir),",
                "llm_max_async = addon_params.get('llm_model_max_async', 2)\n    rag = MegaRAG(\n        working_dir=str(working_dir),\n        llm_model_max_async=llm_max_async,"
            )
        with open(script_path, "w", encoding="utf-8") as f:
            f.write(code)
        print(f"✓ Configured {py_name}")

# C. Patch megarag/llms/openai.py to increase retry attempts on 429 RateLimitError
openai_py_path = megarag_dir / "megarag" / "llms" / "openai.py"
if openai_py_path.exists():
    with open(openai_py_path, "r", encoding="utf-8") as f:
        o_code = f.read()
    if "stop_after_attempt(3)" in o_code:
        o_code = o_code.replace("stop_after_attempt(3)", "stop_after_attempt(12)")
        o_code = o_code.replace(
            "wait_exponential(multiplier=1, min=4, max=10)",
            "wait_exponential(multiplier=1.5, min=2, max=30)"
        )
        with open(openai_py_path, "w", encoding="utf-8") as f:
            f.write(o_code)
        print("✓ Configured resilient OpenAI 429 retry policy")

# D. Patch build_page_assets.py to support hybrid_auto / auto folder discovery
build_assets_path = megarag_dir / "egs" / "utils" / "build_page_assets.py"
if build_assets_path.exists():
    with open(build_assets_path, "r", encoding="utf-8") as f:
        b_code = f.read()
    if "middle_candidates" not in b_code:
        b_code = b_code.replace(
            '    filename = wdir.parent.name if wdir.name == "auto" else wdir.name\n\n    middle_json = wdir / f"{filename}_middle.json"\n    txt_json = wdir / f"{filename}_content_list.json"\n    img_root = wdir / "images"\n    page_img_root = wdir / "page_images"',
            '    filename = wdir.parent.name if wdir.name in ["auto", "hybrid_auto", "vlm_auto", "pipeline"] else wdir.name\n    middle_candidates = list(wdir.glob("*_middle.json")) or list(wdir.parent.glob("*_middle.json"))\n    middle_json = middle_candidates[0] if middle_candidates else (wdir / f"{filename}_middle.json")\n    txt_candidates = list(wdir.glob("*_content_list.json")) or list(wdir.parent.glob("*_content_list.json"))\n    txt_json = txt_candidates[0] if txt_candidates else (wdir / f"{filename}_content_list.json")\n    img_root = wdir / "images" if (wdir / "images").exists() else (wdir.parent / "images")\n    page_img_root = wdir / "page_images" if (wdir / "page_images").exists() else (wdir.parent / "page_images")'
        )
        with open(build_assets_path, "w", encoding="utf-8") as f:
            f.write(b_code)
        print("✓ Configured build_page_assets.py for auto folder discovery")

# E. Configure addon_params.yaml for Kaggle GPU stability
yaml_configs = list(megarag_dir.glob("**/addon_params.yaml"))
for ycfg in yaml_configs:
    with open(ycfg, "r", encoding="utf-8") as f:
        y_text = f.read()
    y_text = re.sub(r"embed_parallel_limit:\s*\d+", "embed_parallel_limit: 1", y_text)
    y_text = re.sub(r"insert_batch_size:\s*\d+", "insert_batch_size: 4", y_text)
    y_text = re.sub(r"entity_extract_max_gleaning:\s*\d+", "entity_extract_max_gleaning: 1", y_text)
    if "llm_model_max_async" not in y_text:
        y_text += "\nllm_model_max_async: 2\n"
    else:
        y_text = re.sub(r"llm_model_max_async:\s*\d+", "llm_model_max_async: 2", y_text)
    if "visual_element_or_map" not in y_text and "entity_types:" in y_text:
        y_text = y_text.replace("  - textbook_structure", "  - textbook_structure\n  - visual_element_or_map")
    with open(ycfg, "w", encoding="utf-8") as f:
        f.write(y_text)
    print(f"✓ Configured MMKG parameters in {ycfg.name}")


## 5. Prepare Example Data (`world_history_tiny`)

Load `World_History_Volume_1.pdf` and `queries.txt` from `/kaggle/input/datasets` (or `/kaggle/input/`) into the workspace `data/` directory (alongside `LightRAG`, `MegaRAG`, `MinerU`).


In [ ]:
import os
import shutil
from pathlib import Path

# Setup data directory directly under REPO_DIR (alongside LightRAG, MegaRAG, MinerU)
data_dir = REPO_DIR / "data"
data_dir.mkdir(parents=True, exist_ok=True)

pdf_file = data_dir / "World_History_Volume_1.pdf"
queries_file = data_dir / "queries.txt"

# 1. Search for dataset files in /kaggle/input/datasets, /kaggle/input, or existing repo paths
kaggle_input = Path("/kaggle/input")
search_roots = [
    kaggle_input / "datasets",
    kaggle_input,
    megarag_dir / "egs" / "world_history_tiny" / "data",
]

pdf_source = None
queries_source = None

for root in search_roots:
    if root.exists():
        # Look for PDF file
        if not pdf_source:
            pdf_matches = list(root.glob("**/World_History_Volume_1.pdf"))
            if pdf_matches:
                pdf_source = pdf_matches[0]
        # Look for queries file
        if not queries_source:
            query_matches = list(root.glob("**/queries.txt"))
            if query_matches:
                queries_source = query_matches[0]

# Copy PDF to data directory
if pdf_source and pdf_source.exists():
    shutil.copy2(pdf_source, pdf_file)
    print(f"✓ Found and copied PDF from: {pdf_source}")
elif pdf_file.exists():
    print(f"✓ PDF already present in data directory: {pdf_file}")
else:
    print(f"⚠️ PDF file 'World_History_Volume_1.pdf' not found in /kaggle/input/datasets!")
    print(f"   Please make sure the dataset is added to the Kaggle notebook input.")

# Copy or generate queries.txt
if queries_source and queries_source.exists():
    shutil.copy2(queries_source, queries_file)
    print(f"✓ Found and copied queries from: {queries_source}")
elif not queries_file.exists():
    sample_queries = """- Question 1: What is the Byzantine Empire?
- Question 2: When did the Roman Empire fall?
- Question 3: Who was Alexander the Great?
- Question 4: What was the Silk Road?
- Question 5: Describe ancient Egyptian civilization.
"""
    with open(queries_file, "w", encoding="utf-8") as f:
        f.write(sample_queries.strip() + "\n")
    print(f"✓ Created benchmark queries file at: {queries_file}")
else:
    print(f"✓ Queries file ready: {queries_file}")

if pdf_file.exists():
    print(f"✓ Input PDF ready: {pdf_file} ({pdf_file.stat().st_size / (1024 * 1024):.2f} MB)")


## 6. Build Multimodal Knowledge Graph (MMKG)

Outputs and intermediate assets are stored inside the run directory `<pdf_name>_run/` located directly under `REPO_DIR` (alongside `LightRAG`, `MegaRAG`, `MinerU`), containing `dumps/` and `exp/`.

We split the MMKG construction into 4 distinct steps:
- **6.1**: Parse PDF using MinerU (`magic-pdf`) $\rightarrow$ `<pdf_name>_run/dumps/`
- **6.2**: Convert PDF pages to images (`pdf2img.py`)
- **6.3**: Build Page Assets manifest (`build_page_assets.py`)
- **6.4**: Construct MMKG (`construct_mmkg.py`) $\rightarrow$ `<pdf_name>_run/exp/`


### 6.1. Parse PDF with MinerU

Extract text, layout, tables, and embedded images from the PDF into `<pdf_name>_run/dumps/`.


In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

pdf_path = data_dir / "World_History_Volume_1.pdf"
pdf_name = pdf_path.stem

# Setup dedicated run directory at the base repo level (alongside LightRAG, MegaRAG, MinerU)
run_dir = REPO_DIR / f"{pdf_name}_run"
dumps_dir = run_dir / "dumps"
dumps_dir.mkdir(parents=True, exist_ok=True)
exp_dir = run_dir / "exp" / pdf_name
exp_dir.mkdir(parents=True, exist_ok=True)
config_file = megarag_dir / "egs" / "world_history_tiny" / "conf" / "addon_params.yaml"

os.chdir(run_dir)
print(f"Run directory (root level): {run_dir}")
print(f"Dumps directory: {dumps_dir}")

# Parse PDF using MinerU magic-pdf CLI (pages 0-9 for quickstart)
print("\n--- [Step 6.1] Parsing PDF with MinerU ---")
magic_pdf_bin = shutil.which("magic-pdf") or f"{sys.executable} -m magic_pdf.tools.cli"
cmd_parse = f"{magic_pdf_bin} -p {pdf_path} -o {dumps_dir} -m auto -e 9"
print(f"Running: {cmd_parse}\n")

# Run directly so all logs, progress bars, and root error messages are shown in full
subprocess.run(cmd_parse, shell=True, check=True)

# Standardize output folders (ensure dumps/pdf_name/auto contains all parsed assets)
doc_dir = dumps_dir / pdf_name
auto_dir = doc_dir / "auto"
auto_dir.mkdir(parents=True, exist_ok=True)

content_lists = list(dumps_dir.rglob("*_content_list.json"))
if content_lists:
    src_dir = content_lists[0].parent
    if src_dir.resolve() != auto_dir.resolve():
        for item in src_dir.iterdir():
            dest = auto_dir / item.name
            if not dest.exists():
                shutil.copytree(item, dest) if item.is_dir() else shutil.copy2(item, dest)

print(f"\n✓ Step 6.1 completed: PDF parsing done!")
print(f"Verified assets under '{auto_dir}':")
for file in sorted(auto_dir.iterdir()):
    print(f"  - {file.name}{'/' if file.is_dir() else f' ({file.stat().st_size / 1024:.1f} KB)'}")


### 6.2. Convert PDF Pages to Images

Convert the PDF pages into JPEG format using `pdf2img.py`.


In [ ]:
import sys
import subprocess
from pathlib import Path
import shutil

print("--- [Step 6.2] Converting PDF pages to images ---")
pdf2img_py = megarag_dir / "egs" / "utils" / "pdf2img.py"
doc_dir = dumps_dir / pdf_name

target_img_dirs = [
    doc_dir / "auto" / "page_images",
    doc_dir / "page_images",
]

primary_img_dir = target_img_dirs[0]
primary_img_dir.mkdir(parents=True, exist_ok=True)

cmd_img = f"{sys.executable} {pdf2img_py} {pdf_path} {primary_img_dir} --dpi 150 --jpeg --end-page 10 --jobs 4"
print(f"Running: {cmd_img}")
subprocess.run(cmd_img, shell=True, check=True)

# Synchronize page_images to doc_dir / page_images
for t_dir in target_img_dirs[1:]:
    t_dir.mkdir(parents=True, exist_ok=True)
    for img in primary_img_dir.glob("*.jpg"):
        dest_img = t_dir / img.name
        if not dest_img.exists():
            shutil.copy2(img, dest_img)

print(f"✓ Step 6.2 completed: Saved {len(list(primary_img_dir.glob('*.jpg')))} page images to {primary_img_dir}")


### 6.3. Build Page Assets Manifest

Merge the parsed text, tables, figure images, and page images into a unified `pages_content.json` manifest.


In [ ]:
import sys
import subprocess
from pathlib import Path
import shutil

print("--- [Step 6.3] Building Page Assets Manifest ---")
build_assets_py = megarag_dir / "egs" / "utils" / "build_page_assets.py"
doc_dir = dumps_dir / pdf_name

# 1. Discover working directory containing parsed content
candidate_dirs = [
    doc_dir / "auto",
    doc_dir / "ocr",
    doc_dir / "txt",
    doc_dir,
]
found_lists = list(dumps_dir.rglob("*_content_list.json"))
for cl in found_lists:
    if cl.parent not in candidate_dirs:
        candidate_dirs.insert(0, cl.parent)

working_dir = None
for c_dir in candidate_dirs:
    if c_dir.exists() and list(c_dir.glob("*_content_list.json")):
        working_dir = c_dir
        break

if not working_dir:
    working_dir = doc_dir / "auto"

# 2. Ensure page_images exists inside working_dir
if not (working_dir / "page_images").exists():
    for pimg_dir in [doc_dir / "page_images", doc_dir / "auto" / "page_images"]:
        if pimg_dir.exists() and pimg_dir != (working_dir / "page_images"):
            shutil.copytree(pimg_dir, working_dir / "page_images")
            break

page_manifest = doc_dir / "pages_content.json"

cmd_assets = f"{sys.executable} {build_assets_py} --working-dir {working_dir} --output {page_manifest}"
print(f"Running: {cmd_assets}")
subprocess.run(cmd_assets, shell=True, check=True)
print(f"✓ Step 6.3 completed: Page assets manifest generated at {page_manifest} ({page_manifest.stat().st_size / 1024:.1f} KB)")


### 6.4. Construct Multimodal Knowledge Graph

Extract multimodal entities, relationships, and embeddings to build the Knowledge Graph using `construct_mmkg.py` into `<pdf_name>_run/exp/`.


In [ ]:
import sys
import time
import subprocess
from pathlib import Path

print("--- [Step 6.4] Constructing Multimodal Knowledge Graph ---")
construct_mmkg_py = megarag_dir / "egs" / "utils" / "construct_mmkg.py"

cmd_mmkg = f"{sys.executable} {construct_mmkg_py} --config-file {config_file} --working-dir {exp_dir} --input-dir {page_manifest}"
print(f"Running: {cmd_mmkg}")
t0 = time.time()
subprocess.run(cmd_mmkg, shell=True, check=True)
print(f"\n✓ Step 6.4 completed: MMKG Construction finished in {time.time() - t0:.1f} seconds!")


## 7. Query with MegaRAG

Following Step 4 of MegaRAG README (`run_quering.sh`): Query the Multimodal Knowledge Graph using `query_mmkg.py`.


In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

query_script = megarag_dir / "egs" / "utils" / "query_mmkg.py"
results_dir = exp_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)
results_file = results_dir / "results.json"

cmd_query = f"""{sys.executable} {query_script} \
    --config-file {config_file} \
    --working-dir {exp_dir} \
    --input-queries {queries_file} \
    --output-file {results_file} \
    --concurrency 4
"""

print(f"Running queries:\n{cmd_query}\n")
t0 = time.time()
subprocess.run(cmd_query, shell=True, check=True)
print(f"\n✓ Querying completed in {time.time() - t0:.1f} seconds!")


## 8. View Results & Knowledge Graph Analysis


In [ ]:
import json
from pathlib import Path
import networkx as nx

# 1. Display Query Results
if results_file.exists():
    with open(results_file, "r", encoding="utf-8") as f:
        results = json.load(f)

    print("=" * 80)
    print("MEGARAG QUERY RESULTS")
    print("=" * 80)

    if isinstance(results, list):
        for i, res in enumerate(results, 1):
            print(f"\n{'─' * 80}")
            print(f"Query {i}: {res.get('query', 'N/A')}")
            print(f"{'─' * 80}")
            if "answer" in res:
                print(f"Answer:\n{res['answer']}\n")
            if "retrieved_context" in res:
                print(f"Retrieved Context:\n{res['retrieved_context'][:300]}...\n")
    elif isinstance(results, dict):
        for query, ans in results.items():
            print(f"\n{'─' * 80}")
            print(f"Query: {query}")
            print(f"{'─' * 80}")
            answer_text = ans.get("answer", ans) if isinstance(ans, dict) else ans
            print(f"Answer:\n{answer_text}\n")
    print("=" * 80)

# 2. Knowledge Graph Analysis
graphml_files = list(exp_dir.glob("**/*.graphml"))
if graphml_files:
    G = nx.read_graphml(graphml_files[0])
    print(f"\nKnowledge Graph Overview ({graphml_files[0].name}):")
    print(f"  • Total Entities (Nodes): {G.number_of_nodes()}")
    print(f"  • Total Relationships (Edges): {G.number_of_edges()}")
    
    degrees = dict(G.degree())
    top_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:5]
    print("\nTop 5 Most Connected Entities:")
    for rank, (node, deg) in enumerate(top_nodes, 1):
        entity_type = G.nodes[node].get("entity_type", "entity")
        print(f"  {rank}. [{entity_type}] {node} ({deg} connections)")
